<a href="https://colab.research.google.com/github/project-ccap/project-ccap.github.io/blob/master/2026notebooks/2026_0614sbert_ccap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    import fugashi
except ImportError:
    !pip install "fugashi[unidic-lite]"


In [ ]:
import torch
from transformers import AutoTokenizer, BertModel  # <- AutoTokenizer をインポート
from scipy.stats import pearsonr

class SentenceBertJapanese:
    def __init__(self, model_name_or_path, device=None):
        # 💡 ここを BertJapaneseTokenizer から AutoTokenizer に変更
        self.tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
        self.model = BertModel.from_pretrained(model_name_or_path)
        self.model.eval()

        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = torch.device(device)
        self.model.to(device)

    def _mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    @torch.no_grad()
    def encode(self, sentences, batch_size=8):
        all_embeddings = []
        iterator = range(0, len(sentences), batch_size)
        for batch_idx in iterator:
            batch = sentences[batch_idx:batch_idx + batch_size]

            # 💡 self.tokenizer.batch_encode_plus(...) を self.tokenizer(...) に変更
            encoded_input = self.tokenizer(
              batch,
              padding="longest",
              truncation=True, return_tensors="pt").to(self.device)

            model_output = self.model(**encoded_input)
            sentence_embeddings = self._mean_pooling(
              model_output,
              encoded_input["attention_mask"]).to('cpu')

            all_embeddings.extend(sentence_embeddings)

        return torch.stack(all_embeddings)

MODEL_NAME = "sonoisa/sentence-bert-base-ja-mean-tokens-v2"
model = SentenceBertJapanese(MODEL_NAME)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:

sentences = ["暴走したAI", "暴走した人工知能"]

sentence_embeddings = model.encode(sentences, batch_size=8)
##print("Sentence embeddings:", sentence_embeddings)

print(f'相関係数:{pearsonr(sentence_embeddings[0], sentence_embeddings[1])[0]:.3f}')

相関係数:0.487


In [ ]:
import sys
MrS_pairs = [('爪を深く切って欲しい', 'ぼごっとしてください'), ('マッサージをしている', 'もみもみしよらす'), ('いちゃいちゃしていたのではないのか', 'ちくちくしよったとだろ'), ('かっこいいですね', 'ばりっとしとる'), ('もやもやする？', 'もじゃもじゃする'), ('異性といちゃいちゃ(こそこそ)していたのではないか', 'こちょこちょしよったとでしょ'), ('シャキッとしている', 'ばりっとしとらす'), ('しっかり切ってください', 'ぴしゃっとしてください'), ('しっかり切ってくれるからよい', 'ぴしゃっとしてくれるけんよか'), ('どんどん（速度？）しているから大丈夫です', 'びゃんびゃんするけんよか'), ('眠りそうになっていた', 'うつらうつらしよった'), ('爪はしっかり切っておかないといけませんよ', 'せんせい！ばりっとしとかなんですよ'), ('自由に寝てもよいですよ？', 'ぶらっと寝なっせ'), ('シャッターを切ってください', 'ばちっと撮ってください'), ('カレンダーをめくらなければいけませんよ', 'ぱくっとせなんっですよ'), ('鉄砲で撃ちますよ', 'ぱんぱんぱんぱんてするですよ'), ('効果音？', 'ばぶっとせなん'), ('眠ってるんだろう', 'ぐーぐーしよらすたい'), ('かっこいい', 'ばりっとしとらす'), ('かっこいい（上をさらに強調して）', 'ばりばりしとらす'), ('言葉がすぐに出るならいいのに', 'ぱっとでるならよかばってん'), ('ちゃんと大便が出ないといけない', 'ぴしゃっと出らないかん'), ('大便が少しだけ出た', 'ちょこっと出た'), ('しっかり覚えている', 'びゃんびゃん覚えとる'), ('電話しようとするがボタンと押すのが間に合わず切れてしまう', 'ぷわーぷわーってなるとたい'), ('とてもたまる', 'びゃんびゃんたまる'), ('怒りたくなる？いらいら？', 'ぐらぐらしよごたる'), ('激しく文句を言いたい', 'ぎゃーぎゃー言おごたる'), ('大便がたくさん出た', 'がぼがぼ出た'), ('激しく文句を言いたい', 'ぎゃーぎゃー言おごたる'), ('時々は行くけれども', 'ちょこちょこは行くばってん'), ('びゃんびゃんするとは＝PT訓練', 'びゃんびゃんするとはなんだったかな'), ('あまりないから（ポツポツしかないから？）', 'ぼとぼとしかなかけん'), ('思い切り切ってよい', 'ばくっと切ってよか'), ('言葉が出たかと思うと続けて出てこない', 'ぱかって出るかと思うとばばっと出ない'), ('寝ている', 'ねんねしとらす'), ('きちんとしないといけない', 'ぴしゃってせなんとたい'), ('元気だけどとても元気というわけではない', 'げんきばってんびゃんびゃんはなか'), ('激しく嘔吐した', 'うぇーうぇー吐いた'), ('どんどんしないと', 'ぎゃんぎゃんせなん'), ('こういう風に・・・と使います', 'こがんしてがーがーぱっぱてします'), ('どんどん言えるもん', 'びゃんびゃん言いきるもん'), ('どんどんは言えないけれども', 'びゃんびゃんは言いきらんばってん'), ('すぱすぱ吸っていた？', 'ぱこぱこ吸いよった'), ('くしゃみが出る', 'へくしゃんの出る'), ('太ったんでしょ，いや，間違えた', 'ぽっちゃりでしょ，あら，しもた'), ('あら，ばきっといった', 'あら，ばぐっていうた'), ('少しだもんな', 'ぼそっとだもんな'), ('ぱっとでないんですよ', 'ばっとでらんとですよ'), ('？', 'ばりっとしとらんけん'), ('くしゃみが出なかった', 'はくしょんのでらんかった'), ('みそしるが??しているのがいいですね', 'みそしるのぱっとしとるのがよかですね'), ('昔は煙草をよく吸っていた', '昔はぱっかぱっか吸いよったたい'), ('きれいにしていないのはきらいだ', 'ぴしゃっとしとかんとすかん'), ('またボールを回す課題をしないといけない', 'またぱかぱかぱかぱかせなんですよ'), ('どんどん発言しないといけない', 'びゃんびゃんせなん'), ('どんどんしてもらったから', 'びゃんびゃんしてもろうたけん'), ('すぐにはわからないですよ，あれを見ても', 'ぱっとなわからんですよ，あれば見ても'), ('くしゃみをされましたよ', 'へくしゃんてしなはったばい'), ('涙がだらっと出る', '涙がだらっと出る'), ('たくさんぼろぼろと出る', 'たいぎゃなぼろぼろ出る'), ('とてもよい（調子）です', 'びゃんびゃんよかです'), ('げっそり痩せた', 'ごっそりなった'), ('（床が）つるつるしてるではないですか', 'つるつるしとるじゃなかですか'), ('つるつるしていたらいけませんよ', 'つるつるしとったらいかんですよ'), ('ふとした時が出ないんですよ', 'ぱっとしたときが出らんとですよ'), ('にこにこしておられる', 'にこにこしとらす'), ('倒れますかね', 'ばたってなりますかね'), ('ごわごわしている？', 'ぶわぶわしとる')]

In [ ]:
for pair in MrS_pairs:
    print(pair, end="")
    sentence_embeddings = model.encode(pair, batch_size=2)
    print(f'相関係数:{pearsonr(sentence_embeddings[0], sentence_embeddings[1])[0]:.3f}')



('爪を深く切って欲しい', 'ぼごっとしてください')相関係数:0.312
('マッサージをしている', 'もみもみしよらす')相関係数:0.242
('いちゃいちゃしていたのではないのか', 'ちくちくしよったとだろ')相関係数:0.518
('かっこいいですね', 'ばりっとしとる')相関係数:0.327
('もやもやする？', 'もじゃもじゃする')相関係数:0.466
('異性といちゃいちゃ(こそこそ)していたのではないか', 'こちょこちょしよったとでしょ')相関係数:0.448
('シャキッとしている', 'ばりっとしとらす')相関係数:0.549
('しっかり切ってください', 'ぴしゃっとしてください')相関係数:0.402
('しっかり切ってくれるからよい', 'ぴしゃっとしてくれるけんよか')相関係数:0.390
('どんどん（速度？）しているから大丈夫です', 'びゃんびゃんするけんよか')相関係数:0.269
('眠りそうになっていた', 'うつらうつらしよった')相関係数:0.431
('爪はしっかり切っておかないといけませんよ', 'せんせい！ばりっとしとかなんですよ')相関係数:0.258
('自由に寝てもよいですよ？', 'ぶらっと寝なっせ')相関係数:0.476
('シャッターを切ってください', 'ばちっと撮ってください')相関係数:0.313
('カレンダーをめくらなければいけませんよ', 'ぱくっとせなんっですよ')相関係数:0.107
('鉄砲で撃ちますよ', 'ぱんぱんぱんぱんてするですよ')相関係数:0.353
('効果音？', 'ばぶっとせなん')相関係数:0.423
('眠ってるんだろう', 'ぐーぐーしよらすたい')相関係数:0.188
('かっこいい', 'ばりっとしとらす')相関係数:0.356
('かっこいい（上をさらに強調して）', 'ばりばりしとらす')相関係数:0.469
('言葉がすぐに出るならいいのに', 'ぱっとでるならよかばってん')相関係数:0.472
('ちゃんと大便が出ないといけない', 'ぴしゃっと出らないかん')相関係数:0.523
('大便が少しだけ出た', 'ちょこっと出た')相関係数:0.480
('しっかり覚えている', 'びゃんびゃん覚えとる')相関係数:0.578
('電